<a href="https://colab.research.google.com/github/srishtiii28/flight_delay_predictor/blob/main/flight_delay_predictor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd

file_path = '/content/drive/MyDrive/2018.csv'
df = pd.read_csv(file_path,nrows=7000)
df.to_csv("2018_final.csv")


# try:
#   df = pd.read_csv(file_path)
#   print(f"Successfully loaded data from {file_path}. Showing first 5 rows:")
#   display(df.head())
# except FileNotFoundError:
#   print(f"Error: File not found at {file_path}. Please check the path and make sure the file exists in your Google Drive.")
# except Exception as e:
#   print(f"An error occurred while loading the file: {e}")

In [ ]:
# Fill NaN values in specific delay columns with 0
delay_cols = ['CARRIER_DELAY', 'WEATHER_DELAY', 'NAS_DELAY', 'SECURITY_DELAY', 'LATE_AIRCRAFT_DELAY']
df[delay_cols] = df[delay_cols].fillna(0)
print("Filled NaN values in delay columns with 0.")

# Fill NaN values in CANCELLATION_CODE with 'N/A'
df['CANCELLATION_CODE'] = df['CANCELLATION_CODE'].fillna('N/A')
print("Filled NaN values in 'CANCELLATION_CODE' with 'N/A'.")

# Display missing values again to see the effect of these changes
print('\nMissing values per column after filling some NaNs:')
display(df.isnull().sum())

Filled NaN values in delay columns with 0.
Filled NaN values in 'CANCELLATION_CODE' with 'N/A'.

Missing values per column after filling some NaNs:


,0
FL_DATE,0
OP_CARRIER,0
OP_CARRIER_FL_NUM,0
ORIGIN,0
DEST,0
CRS_DEP_TIME,0
DEP_TIME,131
DEP_DELAY,147
TAXI_OUT,139
WHEELS_OFF,139


In [ ]:
# --- Added preprocessing fixes ---
# Remove unnamed columns
df = df.loc[:, ~df.columns.str.contains('^Unnamed')]

# Convert date
df['FL_DATE'] = pd.to_datetime(df['FL_DATE'])
df['DAY_OF_WEEK'] = df['FL_DATE'].dt.dayofweek
df['MONTH'] = df['FL_DATE'].dt.month
df = df.drop('FL_DATE', axis=1)

# Remove leakage columns
leakage_cols = [
    'CARRIER_DELAY', 'WEATHER_DELAY', 'NAS_DELAY',
    'SECURITY_DELAY', 'LATE_AIRCRAFT_DELAY'
]
df = df.drop(columns=leakage_cols, errors='ignore')


Once your Drive is mounted, you can access your files. For example, if you have a CSV file named `my_dataset.csv` in the root of your Drive, you can load it into a pandas DataFrame like this:

In [ ]:
# Drop the 'Unnamed: 27' column as it's entirely empty
df = df.drop(columns=['Unnamed: 27'])
print("Dropped 'Unnamed: 27' column.")

# Convert 'FL_DATE' to datetime objects
df['FL_DATE'] = pd.to_datetime(df['FL_DATE'])
print("Converted 'FL_DATE' to datetime.")

# Display info again to confirm changes
print('\nDataFrame Info after initial cleaning:')
df.info()

KeyError: "['Unnamed: 27'] not found in axis"

In [ ]:
print('DataFrame Info:')
df.info()

In [ ]:
print('\nDataFrame Description (Numerical Columns):')
display(df.describe())

In [ ]:
df_filtered = df

In [ ]:
# Drop rows with remaining missing values in key columns for delay prediction
df_filtered.dropna(subset=['DEP_DELAY', 'ARR_DELAY', 'ACTUAL_ELAPSED_TIME', 'AIR_TIME'], inplace=True)

print(f"Flights after dropping rows with remaining NaNs: {len(df_filtered)}")

# Display missing values again to confirm
print('\nMissing values per column after final NaN removal:')
display(df_filtered.isnull().sum())

### Handling Cancelled and Diverted Flights

For a flight delay predictor, we typically want to predict the delay for flights that actually operate and arrive. Therefore, we'll remove all rows where the flight was cancelled or diverted.

In [ ]:
# Before dropping, let's see how many cancelled/diverted flights we have
print(f"Total flights: {len(df)}")
print(f"Cancelled flights: {df['CANCELLED'].sum()}")
print(f"Diverted flights: {df['DIVERTED'].sum()}")

# Filter out cancelled and diverted flights
df_filtered = df[(df['CANCELLED'] == 0) & (df['DIVERTED'] == 0)].copy()

print(f"Flights after removing cancelled/diverted: {len(df_filtered)}")

# Drop the 'CANCELLED' and 'DIVERTED' columns as they are no longer needed in this filtered dataset
df_filtered = df_filtered.drop(columns=['CANCELLED', 'DIVERTED'])
print("Dropped 'CANCELLED' and 'DIVERTED' columns from the filtered DataFrame.")

# Display missing values again for the filtered DataFrame to reassess
print('\nMissing values per column after filtering out cancelled/diverted flights:')
display(df_filtered.isnull().sum())

In [ ]:
print('\nMissing values per column:')
display(df.isnull().sum())

In [ ]:
print('Missing values per column in df_filtered:')
display(df_filtered.isnull().sum())

## train-test-split

In [ ]:
df["delay"] = (df["ARR_DELAY"] > 15).astype(int)

In [ ]:
features = [
    "OP_CARRIER",
    "ORIGIN",
    "DEST",
    "CRS_DEP_TIME",
    "DISTANCE",
    "CRS_ELAPSED_TIME"
]

X = df[features]
y = df["delay"]

In [ ]:
from sklearn.preprocessing import LabelEncoder

encoders = {}

for col in ["OP_CARRIER", "ORIGIN", "DEST"]:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col])
    encoders[col] = le

In [ ]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=150,
    max_depth=8,
    min_samples_split=10,
    min_samples_leaf=5,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)


In [ ]:
joblib.dump(model, "model.pkl")
joblib.dump(encoders, "encoders.pkl")

In [ ]:
y_prob = model.predict_proba(X_test)[:,1]

# try different thresholds
y_pred_03 = (y_prob > 0.3).astype(int)
y_pred_025 = (y_prob > 0.25).astype(int)

In [ ]:
from sklearn.metrics import confusion_matrix,classification_report
y_pred = model.predict(X_test)
print(classification_report(y_pred_03,y_pred_025))

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

cm = confusion_matrix(y_test, y_pred)

disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot()

plt.title("Confusion Matrix")
plt.show()

In [ ]:
from sklearn.metrics import precision_recall_curve

y_prob = model.predict_proba(X_test)[:, 1]

precision, recall, thresholds = precision_recall_curve(y_test, y_prob)

plt.figure()
plt.plot(thresholds, precision[:-1], label="Precision")
plt.plot(thresholds, recall[:-1], label="Recall")

plt.xlabel("Threshold")
plt.ylabel("Score")
plt.title("Precision vs Recall vs Threshold")
plt.legend()
plt.show()

In [ ]:
from sklearn.metrics import roc_curve, auc

fpr, tpr, _ = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)

plt.figure()
plt.plot(fpr, tpr, label=f"AUC = {roc_auc:.2f}")
plt.plot([0, 1], [0, 1], linestyle='--')

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.show()

In [ ]:
import pandas as pd

importances = model.feature_importances_
features = X.columns

feat_imp = pd.Series(importances, index=features).sort_values(ascending=False)

plt.figure()
feat_imp.head(15).plot(kind='barh')

plt.title("Top 15 Feature Importances")
plt.xlabel("Importance")
plt.show()

In [ ]:
from sklearn.tree import plot_tree

plt.figure(figsize=(15, 8))
plot_tree(model.estimators_[0],
          feature_names=X.columns,
          class_names=["On Time", "Delayed"],
          filled=True,
          max_depth=3)

plt.title("Sample Tree from Random Forest")
plt.show()

In [ ]:
import numpy as np

unique, counts = np.unique(y, return_counts=True)

plt.figure()
plt.bar(unique, counts)

plt.xticks([0, 1], ["On Time", "Delayed"])
plt.title("Class Distribution")
plt.ylabel("Count")
plt.show()

In [ ]:
from sklearn.model_selection import learning_curve

train_sizes, train_scores, test_scores = learning_curve(
    model, X, y, cv=3, n_jobs=-1,
    train_sizes=[0.1, 0.3, 0.5, 0.7, 1.0]
)

train_mean = train_scores.mean(axis=1)
test_mean = test_scores.mean(axis=1)

plt.figure()
plt.plot(train_sizes, train_mean, label="Train Score")
plt.plot(train_sizes, test_mean, label="Test Score")

plt.xlabel("Training Size")
plt.ylabel("Score")
plt.title("Learning Curve")
plt.legend()
plt.show()

### Recall vs Threshold Plot

This graph illustrates the relationship between different threshold values and the recall metric for predicting flight delays. Recall measures the model's ability to correctly identify delayed flights (true positives divided by actual positives). As the threshold increases, the model becomes more conservative in predicting delays, which typically decreases recall but may increase precision. This visualization helps in understanding the trade-off between sensitivity (recall) and specificity when adjusting the decision threshold for the flight delay prediction model.

In [ ]:
plt.figure()
plt.plot(thresholds, recall[:-1])

plt.xlabel("Threshold")
plt.ylabel("Recall (Delayed)")
plt.title("Recall vs Threshold")

plt.show()

In [ ]:
!curl ipv4.icanhazip.com

In [ ]:
# Assuming you already ran drive.mount('/content/drive')
!cp /content/model.pkl /content/drive/MyDrive/flight_predict
!cp /content/encoders.pkl /content/drive/MyDrive/flight_predict
!cp /content/2018_final.csv /content/drive/MyDrive/flight_predict

In [ ]:
import joblib
# Replace 'model' and 'encoders' with the variable names from your training code
joblib.dump(model, "model.pkl")
joblib.dump(encoders, "encoders.pkl")

In [ ]:
!import joblib

# These names must match the variables in your flight_delay_predict.ipynb
try:
    joblib.dump(model, "model.pkl")
    joblib.dump(encoders, "encoders.pkl")
    print("✅ SUCCESS: model.pkl and encoders.pkl created in /content/")
except NameError:
    print("❌ ERROR: You must run your training code cells first!")

In [ ]:
%%writefile app.py
import streamlit as st
import numpy as np
import joblib

# Load locally (no Drive paths needed)
model = joblib.load("model.pkl")
encoders = joblib.load("encoders.pkl")

st.title("✈️ Flight Delay Predictor")
carrier = st.selectbox("Airline", encoders["OP_CARRIER"].classes_)
origin = st.selectbox("Origin Airport", encoders["ORIGIN"].classes_)
dest = st.selectbox("Destination Airport", encoders["DEST"].classes_)
crs_dep_time = st.number_input("Scheduled Departure Time (HHMM)", 0, 2359)
distance = st.number_input("Distance", 0)
crs_elapsed = st.number_input("Scheduled Duration (minutes)", 0)

if st.button("Predict"):
    carrier_enc = encoders["OP_CARRIER"].transform([carrier])[0]
    origin_enc = encoders["ORIGIN"].transform([origin])[0]
    dest_enc = encoders["DEST"].transform([dest])[0]
    input_data = np.array([[carrier_enc, origin_enc, dest_enc, crs_dep_time, distance, crs_elapsed]])
    prediction = model.predict(input_data)
    if prediction[0] == 1:
        st.error("⚠️ Flight will be DELAYED")
    else:
        st.success("✅ Flight will be ON TIME")

In [ ]:
!ls /content/

In [ ]:
import joblib
import os

# Create the files in the current working directory
try:
    # Use the variables from your training code
    joblib.dump(model, "model.pkl")
    joblib.dump(encoders, "encoders.pkl")
    print(f"✅ Success! Files created at: {os.getcwd()}")
except NameError as e:
    print(f"❌ Error: {e}. You must run your training code first so the variables exist!")

In [ ]:
import os

files = ["app.py", "model.pkl", "encoders.pkl"]
for f in files:
    if os.path.exists(f):
        print(f"✅ {f} is exactly where it should be.")
    else:
        print(f"❌ {f} is MISSING. The app will fail.")

In [ ]:
# 1. Kill everything
!pkill streamlit
!pkill npx

# 2. Start the App
!nohup python -m streamlit run app.py &

# 3. Get IP Password
print("Your Password IP:")
!curl ipv4.icanhazip.com

# 4. Start Tunnel
!npx localtunnel --port 8501

In [ ]:
!tail -n 20 nohup.out
